In [1]:
import os
import json
import string
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence
from sklearn.metrics import f1_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Thiết bị sử dụng:", device)

Thiết bị sử dụng: cuda


In [ ]:
class Vocab:
    def __init__(self, train_path: str, label_key: str = "topic"):
        self.pad = "<pad>"
        self.unk = "<unk>"
        self.label_key = label_key 

        with open(train_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        all_words = set()
        labels = set()
        
        for item in data:
            sentence = self.preprocess_sentence(item["sentence"])
            all_words.update(sentence.split())
            labels.add(item[label_key])
        self.w2i = {
            self.pad: 0,
            self.unk: 1
        }
        for idx, word in enumerate(sorted(all_words), start=2):
            self.w2i[word] = idx

        self.i2w = {idx: word for word, idx in self.w2i.items()}

        self.l2i = {label: idx for idx, label in enumerate(sorted(labels))}
        self.i2l = {idx: label for label, idx in self.l2i.items()}

    @property
    def n_labels(self) -> int:
        return len(self.l2i)
    
    @property
    def len(self) -> int:
        return len(self.w2i)

    def preprocess_sentence(self, sentence: str) -> str:
        translator = str.maketrans("", "", string.punctuation)
        sentence = sentence.lower()
        sentence = sentence.translate(translator)
        return sentence

    def encode_sentence(self, sentence: str) -> torch.Tensor:
        sentence = self.preprocess_sentence(sentence)
        words = sentence.split()
        if len(words) == 0:
            words = [self.unk]
        unk_id = self.w2i[self.unk]
        word_ids = [self.w2i.get(word, unk_id) for word in words]
        return torch.tensor(word_ids, dtype=torch.long)

    def encode_label(self, label: str) -> torch.Tensor:
        label_idx = self.l2i[label]
        return torch.tensor(label_idx, dtype=torch.long)

In [6]:
class UIT_VSFC(Dataset):
    def __init__(self, path: str, vocab: Vocab):
        super().__init__()
        with open(path, "r", encoding="utf-8") as f:
            self._data = json.load(f)
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self._data)

    def __getitem__(self, idx: int) -> dict:
        item = self._data[idx]
        sentence = item["sentence"]
        label = item[self.vocab.label_key]

        input_ids = self.vocab.encode_sentence(sentence)
        label_idx = self.vocab.encode_label(label)

        return {
            "input_ids": input_ids,  # 1D Tensor: độ dài bằng số từ trong câu này
            "label": label_idx       # Tensor chứa 1 số nguyên (0, 1, 2 hoặc 3)
        }


def collate_fn(items: list[dict]) -> dict:
    input_ids = [item["input_ids"] for item in items]
    
    lengths = torch.tensor([ids.shape[0] for ids in input_ids], dtype=torch.long)
    
    max_len = int(lengths.max())

    padded_inputs = torch.stack([
        F.pad(
            input_tensor, 
            pad=(0, max_len - input_tensor.shape[0]), 
            mode="constant", 
            value=0
        ) for input_tensor in input_ids
    ])
    labels = torch.stack([item["label"] for item in items])

    return {
        "input_ids": padded_inputs,
        "lengths": lengths,
        "label": labels
    }

In [ ]:
class GruModel(nn.Module):
    def __init__(
        self, 
        vocab_size: int,
        hidden_size: int = 256,
        n_layer: int = 5,
        n_labels: int = 4,
        padding_idx: int = 0,
        dropout: float = 0.2
    ):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=hidden_size,
            padding_idx=padding_idx
        )

        self.gru = nn.GRU(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=n_layer,
            batch_first=True,
            dropout=dropout if n_layer > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            in_features=hidden_size,
            out_features=n_labels
        )

    def forward(self, inputs: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded_feats = self.embedding(inputs)
        packed = pack_padded_sequence(
            embedded_feats, 
            lengths.cpu(), 
            batch_first=True, 
            enforce_sorted=False
        )

        _, features = self.gru(packed)
        last_hidden = features[-1]
        last_hidden = self.dropout(last_hidden)
        logits = self.classifier(last_hidden)

        return logits


In [10]:
def train_epoch(
    model: nn.Module, 
    dataloader: DataLoader, 
    optimizer: optim.Optimizer, 
    loss_fn: nn.Module, 
    device: torch.device
) -> float:
    model.train()
    running_loss = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        lengths = batch["lengths"]
        labels = batch["label"].to(device)
        optimizer.zero_grad()
        logits = model(input_ids, lengths)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss.append(loss.item())

    return float(np.mean(running_loss))


@torch.no_grad()
def evaluate(
    model: nn.Module, 
    dataloader: DataLoader, 
    device: torch.device
) -> float:
    model.eval()
    gts = []
    preds = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        lengths = batch["lengths"]
        labels = batch["label"].to(device)

        logits = model(input_ids, lengths)
        predicted_labels = logits.argmax(dim=-1)
        gts.extend(labels.cpu().tolist())
        preds.extend(predicted_labels.cpu().tolist())

    f1 = f1_score(gts, preds, average="macro")
    return f1

In [11]:
import os

TRAIN_PATH = "datasets/UIT-VSFC/UIT-VSFC-train.json"
DEV_PATH   = "datasets/UIT-VSFC/UIT-VSFC-dev.json"
TEST_PATH  = "datasets/UIT-VSFC/UIT-VSFC-test.json"

vocab = Vocab(train_path=TRAIN_PATH, label_key="topic")

train_loader = DataLoader(UIT_VSFC(TRAIN_PATH, vocab), batch_size=32, shuffle=True, collate_fn=collate_fn)
dev_loader   = DataLoader(UIT_VSFC(DEV_PATH, vocab),   batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(UIT_VSFC(TEST_PATH, vocab),  batch_size=32, shuffle=False, collate_fn=collate_fn)

model = GruModel(
    vocab_size=vocab.len,
    hidden_size=256,
    n_layer=5,
    n_labels=vocab.n_labels,
    padding_idx=0,
    dropout=0.2
).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss().to(device)

os.makedirs("checkpoints", exist_ok=True)
CHECKPOINT_PATH = "checkpoints/best_gru.pth"

epoch = 0
best_f1 = 0.0
patience = 0
MAX_PATIENCE = 5

while True:
    epoch += 1
    train_loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
    dev_f1 = evaluate(model, dev_loader, device)

    print(f"Epoch [{epoch:02d}] | Train Loss: {train_loss:.4f} | Dev F1-macro: {dev_f1:.4f}")

    if dev_f1 > best_f1:
        best_f1 = dev_f1
        patience = 0
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"Best F1: {best_f1:.4f}")
    else:
        patience += 1
        print(f"Patience: {patience}/{MAX_PATIENCE}")

    if patience >= MAX_PATIENCE:
        break

model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
test_f1 = evaluate(model, test_loader, device)

print(f"\nKẾT QUẢ ĐÁNH GIÁ CUỐI CÙNG (BÀI 2 - GRU):")
print(f"Test Set Macro F1-score: {test_f1:.4f}")

Epoch [01] | Train Loss: 0.4826 | Dev F1-macro: 0.6759
Best F1: 0.6759
Epoch [02] | Train Loss: 0.3620 | Dev F1-macro: 0.7147
Best F1: 0.7147
Epoch [03] | Train Loss: 0.3198 | Dev F1-macro: 0.7392
Best F1: 0.7392
Epoch [04] | Train Loss: 0.2800 | Dev F1-macro: 0.7507
Best F1: 0.7507
Epoch [05] | Train Loss: 0.2449 | Dev F1-macro: 0.7342
Patience: 1/5
Epoch [06] | Train Loss: 0.2209 | Dev F1-macro: 0.7299
Patience: 2/5
Epoch [07] | Train Loss: 0.2041 | Dev F1-macro: 0.7447
Patience: 3/5
Epoch [08] | Train Loss: 0.1611 | Dev F1-macro: 0.7471
Patience: 4/5
Epoch [09] | Train Loss: 0.1416 | Dev F1-macro: 0.7568
Best F1: 0.7568
Epoch [10] | Train Loss: 0.1221 | Dev F1-macro: 0.7459
Patience: 1/5
Epoch [11] | Train Loss: 0.1061 | Dev F1-macro: 0.7497
Patience: 2/5
Epoch [12] | Train Loss: 0.0975 | Dev F1-macro: 0.7368
Patience: 3/5
Epoch [13] | Train Loss: 0.0847 | Dev F1-macro: 0.7424
Patience: 4/5
Epoch [14] | Train Loss: 0.0883 | Dev F1-macro: 0.7530
Patience: 5/5

KẾT QUẢ ĐÁNH GIÁ CUỐI C